# Optional guarded IonQ Forte-1 reproduction

This notebook reconstructs the frozen 4-qubit H2 GQE circuit and the Pauli
coherence diagnostic `W = X_0 tensor X_1 tensor Y_2 tensor Y_3`, abbreviated
`XXYY`. It is **not** part of the default judge workflow.

Use the qBraid-SDK kernel. Before the first import, install the isolated Forte
requirements:

```python
%pip install --user --upgrade --force-reinstall -r requirements-forte.txt
```

Then **restart the kernel**. NumPy 2.4.6 is required because the tested Numba
build rejects NumPy 2.5. QPU submission remains disabled by default.


## 1. Load and verify the optional environment

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import sys

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / "forte", CWD / "Source_Code" / "forte"]
for parent in CWD.parents:
    candidates.extend([parent / "forte", parent / "Source_Code" / "forte"])
ROOT = next((p for p in candidates if (p / "forte_hardware.py").is_file()), None)
assert ROOT is not None, "Could not locate Source_Code/forte."
sys.path.insert(0, str(ROOT))
import forte_hardware as fh

def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"

print("Folder:", ROOT)
print("qBraid SDK:", version("qbraid"))
print("Qiskit:", version("qiskit"))
print("NumPy:", version("numpy"))
assert version("numpy") == "2.4.6", "Install requirements-forte.txt and restart this kernel."
assert version("qbraid") == "0.12.2", "Use the pinned optional Forte environment."


## 2. Reconstruct the circuit locally — no provider contact

In [ ]:
reference = fh.load_reference(ROOT)
preflight = fh.qiskit_preflight(reference)
hardware_circuit = preflight["hardware_circuit"]
print("Independent NumPy/Qiskit fidelity:", f"{preflight['fidelity_numpy_vs_qiskit']:.16f}")
print("Diagnostic transpilation fidelity:", f"{preflight['diagnostic_fidelity']:.16f}")
print("Diagnostic depth:", preflight["diagnostic_depth"])
print("Diagnostic operations:", preflight["diagnostic_operations"])
print("Circuit SHA-256:", preflight["hardware_circuit_qasm2_sha256"])
print("Ideal <W>, W=X0 tensor X1 tensor Y2 tensor Y3:", f"{preflight['ideal_xxyy_expectation']:.12f}")
print(hardware_circuit.draw(output="text", fold=120))


## 3. Optional qBraid hardware-aware dry run — disabled by default

A safe **Run All** remains fully local. Set `CONTACT_QBRAID=True` only when you
deliberately want to resolve the live device and apply qBraid's transform with
`prepare=False`. This optional dry run cannot create a job.


In [ ]:
CONTACT_QBRAID = False

if not CONTACT_QBRAID:
    print("QBRAID CONTACT DISABLED — local preflight only.")
else:
    from qbraid.runtime import QbraidProvider
    provider = QbraidProvider()
    dry_device = provider.get_device(fh.TARGET_DEVICE_ID)
    snapshot = fh.assert_forte_ready(dry_device)
    print(json.dumps(snapshot, indent=2, sort_keys=True))
    dry_device.set_options(prepare=False)
    try:
        transformed = dry_device.apply_runtime_profile(hardware_circuit)
    finally:
        dry_device.set_options(prepare=True)
    print("Forte transpile/transform/strict-validation dry-run: PASS")
    print("Transformed program type:", type(transformed).__name__)


## 4. Deliberate authorization

Leave both values unchanged for a safe Run All. A new task is stochastic and
recorded a cost of 2,830 qBraid credits. Confirm the current dashboard price and
team account before authorizing. The duplicate-attempt lock is local to this
extracted copy, so never authorize concurrent copies.


In [ ]:
AUTHORIZE_QPU_SUBMISSION = False
CONFIRMATION_PHRASE = ""

print("Device:", fh.TARGET_DEVICE_ID)
print("Shots (locked):", fh.SHOTS)
print("Recorded task credits:", fh.EXPECTED_QPU_CREDITS)
print("Required confirmation phrase:")
print(fh.expected_confirmation())


## 5. Submit at most one task — skipped while locked

In [ ]:
if not AUTHORIZE_QPU_SUBMISSION:
    print("QPU SUBMISSION LOCKED — no job created and no credits spent.")
else:
    assert CONTACT_QBRAID, "Set CONTACT_QBRAID=True before any live submission."
    from qbraid.runtime import QbraidProvider
    fh.assert_submission_guard(AUTHORIZE_QPU_SUBMISSION, CONFIRMATION_PHRASE)
    submit_provider = QbraidProvider()
    submit_device = submit_provider.get_device(fh.TARGET_DEVICE_ID)
    fh.assert_forte_ready(submit_device)
    fh.create_attempt_lock(ROOT, preflight)
    job = submit_device.run(hardware_circuit, shots=fh.SHOTS)
    submission_path = fh.write_submission_record(ROOT, job, submit_device, preflight, reference)
    print("FORTE JOB SUBMITTED")
    print("Job ID:", job.id)
    print("Submission record:", submission_path)


## 6. Retrieve later — skipped if this copy submitted no job

In [ ]:
submission_path = ROOT / fh.RESULTS_DIR / fh.SUBMISSION_RECORD
if not submission_path.is_file():
    print("No local submission record; retrieval skipped.")
elif not CONTACT_QBRAID:
    print("QBRAID CONTACT DISABLED — retrieval skipped.")
else:
    from qbraid.runtime import QbraidJob, QbraidProvider
    submission = json.loads(submission_path.read_text(encoding="utf-8"))
    retrieve_provider = QbraidProvider()
    retrieve_device = retrieve_provider.get_device(fh.TARGET_DEVICE_ID)
    retrieved_job = QbraidJob(submission["job_id"], device=retrieve_device, client=retrieve_provider.client)
    status = retrieved_job.status()
    status_name = str(getattr(status, "name", status)).upper()
    print("Job ID:", retrieved_job.id)
    print("Status:", status_name)
    if "COMPLETED" in status_name:
        hardware_result = retrieved_job.result()
        result_path = fh.write_result_record(
            ROOT, retrieved_job, hardware_result, hardware_result.data.get_counts(), preflight
        )
        print("RESULT SAVED:", result_path)
    else:
        print("No result requested; return after qBraid reports COMPLETED.")
